# Freddie Mac Data Profiling
Here I do the same thing I did with the FRED data. I inspect the data, find the problems, and use them to write the cleaning steps that a PySpark job will run later. As with FRED, I look closely at one year first. I start with 2008 because it was the peak of the financial crisis, and I look at both its origination and performance files. Then I run a quick automated check across all 18 years, and from that I write a clear cleaning plan.

In [1]:
# Load the column names from the config file, so the nameless Freddie Mac columns make sense
from pathlib import Path
import zipfile

import pandas as pd
import yaml


def find_project_root(start=Path.cwd()):
    """Walk up from the current folder until we find the one containing .git."""
    for folder in [start, *start.parents]:
        if (folder / ".git").exists():
            return folder
    raise FileNotFoundError(f"Could not find the project root above {start}")


PROJECT_ROOT = find_project_root()
LANDING = PROJECT_ROOT / "data" / "bronze" / "freddie_mac"

with open(PROJECT_ROOT / "config" / "freddie_mac_columns.yaml") as f:
    layout = yaml.safe_load(f)

orig_cols = [col["name"] for col in layout["origination"]]
perf_cols = [col["name"] for col in layout["performance"]]
print("Origination columns:", len(orig_cols))
print("Performance columns:", len(perf_cols))

Origination columns: 31
Performance columns: 35


## Load the 2008 Origination file. 
I am working with 2008 because it was the peak of the financial crisis. I load every value as text so I can see the raw values before any conversion.

In [ ]:
zip_path = LANDING / "sample_2008.zip"

with zipfile.ZipFile(zip_path) as z:
    print("Files in the zip:", z.namelist())
    with z.open("sample_orig_2008.txt") as f:
        orig_raw = pd.read_csv(f, sep="|", header=None, dtype=str)

print("Shape:", orig_raw.shape)
pd.set_option("display.max_columns", None)
orig_raw.columns = orig_cols
orig_raw.head()

### Initial Inspection of the Loan Origination Data

Each row describes the borrower and the loan, for example whether the borrower is a first time homebuyer, their credit score, the loan amount, the property state and type, how the home is used, the interest rate, the loan term, the loan to value ratio, and the first payment date in YYYYMM format.

The "loan_id" column serves as the identifier for connecting each loan to its corresponding record in the mortgage performance file. 
The "property_state" and "first_payment_date" columns will also be important when connecting the mortgage data to economic indicators from FRED.

During my initial inspection (first five rows), I identified possible codes for unavailable values, including "999" in "orig_dti" and "9999" in "vantage_score". I also noticed that "msa", "pre_harp_loan_id", and "special_eligibility_program" appear to be partially and/or entirely blank, while "seller_name" contains "OTHER" in the first 5 rows. These observations will need further consideration during data cleaning and preparation.

### Column Overview

In [ ]:
# PRofile every column
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

def profile_column(series):
    """Summarize one column and show how many blanks, how many different values, and the most common ones."""
    return {
        "missing": series.isna().sum(),
        "distinct": series.nunique(),
        "top_values": series.value_counts().head(5).to_dict(),
    }

summary = pd.DataFrame({col: profile_column(orig_raw[col]) for col in orig_raw.columns}).T
summary

In [4]:
# count the likely "not available"
numeric_cols = ["credit_score", "mi_pct", "num_units", "orig_cltv", "orig_dti",
                "orig_ltv", "orig_loan_term", "num_borrowers", "vantage_score"]
suspect_codes = ["9", "99", "999", "9999"]

rows = []
for col in numeric_cols:
    counts = {code: (orig_raw[col] == code).sum() for code in suspect_codes}
    rows.append({"column": col, **counts})

pd.DataFrame(rows)

,column,9,99,999,9999
0,credit_score,0,0,0,35
1,mi_pct,1,0,0,0
2,num_units,0,0,0,0
3,orig_cltv,13,4,1,0
4,orig_dti,150,0,1026,0
5,orig_ltv,12,0,1,0
6,orig_loan_term,0,0,0,0
7,num_borrowers,0,23,0,0
8,vantage_score,0,0,0,50000


In [5]:
fred_states = {"AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA",
               "KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM",
               "NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA",
               "WV","WI","WY"}

unmatched = orig_raw[~orig_raw["property_state"].isin(fred_states)]["property_state"].value_counts()
print("Loans in states FRED doesn't cover:")
print(unmatched)
print("Total:", unmatched.sum(), "loans")

Loans in states FRED doesn't cover:
property_state
PR    144
GU     15
VI      5
Name: count, dtype: int64
Total: 164 loans


### Findings from the Full Origination File (2008)

After checking all 50,000 loans, I found a few things that will shape my cleaning plan.

The first five rows were misleading. They showed "OTHER" as the only seller, but the full file has 20 sellers, with Wells Fargo the largest. I learned not to judge the data from the first few rows.

Some values are codes for "not available". 9999 appears in credit_score (35 loans), 999 in orig_dti (1,026 loans, about 2%), 99 in num_borrowers (23 loans), and 9 in first_time_homebuyer_flag (15 loans). These are impossible values for those columns, so they will be treated as missing. Smaller numbers like 9 in orig_dti or orig_ltv could be real, so I will confirm each code in the Freddie Mac User Guide first.

Some columns have the same value for every loan in 2008, such as amortization_type, interest_only_flag, harp_indicator, and vantage_score, and two columns are completely blank. I will check the other years before deciding whether to drop them.

Two findings matter for joining with FRED. property_state has 54 values while FRED covers 51, so loans in Puerto Rico (144), Guam (15), and the US Virgin(5), 164 loans in total will not match. msa is blank for about 21% of loans, but this does not affect the join because I join on state.

Next, I will confirm the codes in the User Guide, check which states do not match FRED, and repeat these checks for all 18 years.

### 2008 Performance File

The performance file tracks every loan month by month after it was made. Each row is one loan in one month, showing details such as the balance owed and how many payments behind the loan is. This is the file my delinquency target will come from. It links to the origination file through loan_id.

In [ ]:
# Read only the first line to count columns before loading the whole file
with zipfile.ZipFile(zip_path) as z:
    with z.open("sample_perf_2008.txt") as f:
        first_line = f.readline().decode()

print("Columns in performance file:", first_line.count("|") + 1)
print("First row:", first_line)

In [ ]:
# Load all rows and all 35 columns, naming them from the config file
with zipfile.ZipFile(zip_path) as z:
    with z.open("sample_perf_2008.txt") as f:
        perf_raw = pd.read_csv(f, sep="|", header=None, names=perf_cols, dtype=str)

print("Shape:", perf_raw.shape)
perf_raw.head(10)

The full 2008 performance file has 2,459,814 rows and 35 columns which uses about 0.94GB of memory. All 18 years together have about 56 million rows, which would need roughly 20 GB, more than my laptop's total memory. This is why I profile one year locally in pandas and process the full dataset with Spark in AWS Glue.

In [8]:
# Big picture checks for size, number of loans, time span, and delinquency codes
print("Rows:", len(perf_raw))
print("Distinct loans:", perf_raw["loan_id"].nunique())
print("Months covered:", perf_raw["period"].min(), "to", perf_raw["period"].max())
print()
print("Delinquency status values:")
print(perf_raw["delinquency_status"].value_counts().head(15))

Rows: 2459814
Distinct loans: 50000
Months covered: 200801 to 202606

Delinquency status values:
delinquency_status
00    2299341
01      56528
02      19780
RA      10091
03       9595
04       7062
05       5983
06       5056
07       4360
08       3859
09       3462
10       3039
11       2697
12       2416
13       2178
Name: count, dtype: int64


### First Look at the Performance File

The 2008 performance file has 2,459,814 rows, which matches the count recorded when I uploaded it to S3, so no data was lost along the way. It covers all 50,000 loans from the origination file, with no extra loans, so every loan can be linked between the two files. The data runs from January 2008 to June 2026, the cutoff of this release.

The delinquency_status column shows how many payments behind each loan was in each month. 00 means the loan was current, 01 means 30 to 59 days behind, 02 means 60 to 89 days behind, and 03 means 90 to 119 days behind. Higher numbers mean the loan fell further behind, and the Freddie Mac User Guide says the value is capped at 99. Most rows (about 93.5%) are 00, which means most loans were paid on time in most months.

I found four things to handle in cleaning. First, the codes are stored as two digit text like "03", so they need to be converted to numbers carefully. Second, the code "RA" is not a number. The User Guide confirms it means REO Acquisition, where the lender took the property through foreclosure. Since this is an even worse outcome than being 90 days behind, it must count as serious delinquency and must not be lost when converting the codes to numbers. The guide also lists "XX" for a value that is not available, which should be treated as missing, not as a delinquency. Third, for periods on or before September 2025, 00 can also mean the month before the loan's first payment was due. This explains why some loans have rows before their first payment date. Fourth, because of this, the 24 month window has to start at each loan's first payment date, not at the first row in the file.

This gives me my target rule which is, a loan is seriously delinquent if within its first 24 months after the first payment date, its status reaches 03 or higher, or RA.

In [9]:
# Find every delinquency status value that is not purely digits
statuses = perf_raw["delinquency_status"]
non_numeric = statuses[~statuses.str.isdigit()].value_counts(dropna=False)
print("Non numeric status codes:")
print(non_numeric)
print()
print("Highest numeric status:", statuses[statuses.str.isdigit()].astype(int).max())

Non numeric status codes:
delinquency_status
RA    10091
Name: count, dtype: int64

Highest numeric status: 99


### Checking the Non Numeric Codes

The only non numeric code in the 2008 file is RA, which appears in 10,091 monthly rows. There is no XX and there are no blank values, so every month has a usable status. The highest numeric status is 99, which matches the cap described in the User Guide. Some loans were 99 or more payments behind, which shows how long some 2008 loans stayed delinquent during the crisis. Values this high do not change my target, since any status of 03 or more already counts as serious delinquency.

In [10]:
ra_rows = perf_raw[perf_raw["delinquency_status"] == "RA"]
print("Loans with at least one RA month:", ra_rows["loan_id"].nunique())
print("Average RA months per loan:", round(len(ra_rows) / ra_rows["loan_id"].nunique(), 1))
print()
print("Loan age at first RA (months):")
first_ra_age = ra_rows.groupby("loan_id")["loan_age"].apply(lambda s: s.astype(int).min())
print(first_ra_age.describe())

Loans with at least one RA month: 1186
Average RA months per loan: 8.5

Loan age at first RA (months):
count    1186.000000
mean       45.046374
std        28.103823
min         5.000000
25%        26.000000
50%        39.000000
75%        56.000000
max       204.000000
Name: loan_age, dtype: float64


### Loans That Reached REO

1,186 loans reached RA, about 2.4% of the 2008 vintage. Each one shows RA for about 8.5 months on average, because the code stays on the loan until the property is sold. That is why I count loans, not rows.

Most loans reached RA after their first 24 months. The median is month 39, and only about a quarter reached it by month 26. A few reached it as early as month 5, so RA can still fall inside the target window.

In [11]:
perf = perf_raw[["loan_id", "loan_age", "delinquency_status"]].copy()
perf["loan_age"] = perf["loan_age"].astype(int)
first24 = perf[perf["loan_age"] <= 24]

status = first24["delinquency_status"]
status_num = pd.to_numeric(status, errors="coerce")   # RA becomes NaN here

hit_03 = (status_num >= 3).groupby(first24["loan_id"]).any()
hit_ra = (status == "RA").groupby(first24["loan_id"]).any()
target = hit_03 | hit_ra

print("Loans seriously delinquent in first 24 months:", target.sum())
print("Target rate:", round(target.mean() * 100, 2), "%")
print("Loans flagged only because of RA:", (hit_ra & ~hit_03).sum())

Loans seriously delinquent in first 24 months: 2235
Target rate: 4.47 %
Loans flagged only because of RA: 0


The rule produces a sensible 4.47% target rate for 2008, RA adds no extra positives, and the low rate means the models must be evaluated with imbalance aware metrics.

In [12]:
def to_months(yyyymm):
    yyyymm = yyyymm.astype(int)
    return (yyyymm // 100) * 12 + (yyyymm % 100)

check = perf_raw[["loan_id", "period", "loan_age"]].merge(
    orig_raw[["loan_id", "first_payment_date"]], on="loan_id"
)
check["months_since_first_pay"] = to_months(check["period"]) - to_months(check["first_payment_date"])
check["difference"] = check["loan_age"].astype(int) - check["months_since_first_pay"]

print(check["difference"].value_counts().head())

difference
 1     2335591
-21       3492
-23       3370
-24       3114
-20       2994
Name: count, dtype: int64


### How Loan Age Is Counted

The User Guide defines loan age as the number of months since the first payment date plus one, so loan age 1 is the first payment month. My check confirms this for about 95% of rows. The rest are modified loans, where the guide says loan age restarts from the modification date. Because of this, loan age is not a reliable way to measure the first 24 months for modified loans. I will measure the target window from first_payment_date instead, which never resets.

In [13]:
check = perf_raw[["loan_id", "period", "delinquency_status"]].merge(
    orig_raw[["loan_id", "first_payment_date"]], on="loan_id"
)
check["months_since_first_pay"] = to_months(check["period"]) - to_months(check["first_payment_date"])
window = check[check["months_since_first_pay"].between(0, 23)]

status = window["delinquency_status"]
status_num = pd.to_numeric(status, errors="coerce")
hit_03 = (status_num >= 3).groupby(window["loan_id"]).any()
hit_ra = (status == "RA").groupby(window["loan_id"]).any()
target_fixed = hit_03 | hit_ra

print("Old target (loan_age):", target.sum())
print("New target (first payment date):", target_fixed.sum())
print("Target rate:", round(target_fixed.mean() * 100, 2), "%")

Old target (loan_age): 2235
New target (first payment date): 1984
Target rate: 3.98 %


### Corrected Target

Measuring the window from first_payment_date instead of loan age changed the number of positive loans from 2,235 to 1,984. The 251 loans that dropped out were modified loans that became 90 days late after their real first 24 months, but looked like they were inside the window because their loan age had restarted. The corrected target rate for the 2008 vintage is 3.98%. From here on, the target window is measured from first_payment_date.

### Zero Balance codes
This checks why loans left the data. A loan that paid off in month 10 could not go 90 days late in month 20, so I need to see how many loans left before month 24 and decide how to label them.

In [14]:
last_rows = perf_raw.dropna(subset=["zero_balance_code"])
print("Loans with a zero balance code:", last_rows["loan_id"].nunique())
print()
print(last_rows["zero_balance_code"].value_counts())

Loans with a zero balance code: 48912

zero_balance_code
01    45708
09     1175
16      689
03      544
96      380
02      293
15      123
Name: count, dtype: int64


In [15]:
zb = perf_raw.dropna(subset=["zero_balance_code"])[["loan_id", "zero_balance_code", "zero_balance_date"]]
zb = zb.merge(orig_raw[["loan_id", "first_payment_date"]], on="loan_id")
zb["months_to_exit"] = to_months(zb["zero_balance_date"]) - to_months(zb["first_payment_date"])
zb["left_before_24"] = zb["months_to_exit"] < 24
zb["target"] = zb["loan_id"].map(target_fixed).fillna(False)

print(pd.crosstab(zb["zero_balance_code"], [zb["left_before_24"], zb["target"]]))

left_before_24     False        True       
target             False True   False True 
zero_balance_code                          
01                 28021   417  17196    74
02                   193    90      0    10
03                   336   160      2    46
09                   563   497      0   115
15                    76    47      0     0
16                   446   243      0     0
96                   109   168     42    61


### Loans That Left Before 24 Months

I checked whether loans that left the data early have a reliable label. Loans that ended in foreclosure, short sale, or charge off (02, 03, 09) before month 24 were almost all already positive. Only 2 were not, so I count these endings inside the window as positive too. No loans sold by Freddie Mac (15, 16) left before month 24, so their windows are complete. For defect loans (96), 42 left before month 24 without becoming delinquent, so their outcome is unknown and I exclude them. 17,196 loans paid off early without going late. I keep these as negatives because paying off in full is a good outcome, and I note this as a limitation.

Final target rule is, a loan is positive if within 24 months of its first payment date, its status reaches 03 or higher, it shows RA, or it ends with code 02, 03, or 09. The 2008 target rate stays at about 3.98%.
Loans with code 96 that left before month 24 without becoming positive are excluded.

### A quick look at the rest ofthe performance columns
The performance file has 35 columns and I have only studied 4. Before writing the cleaning steps, I do a quick check of all of them.

In [16]:
summary = pd.DataFrame({
    "missing_pct": (perf_raw.isna().mean() * 100).round(1),
    "distinct": perf_raw.nunique(),
    "example": perf_raw.apply(lambda col: col.dropna().iloc[0] if col.notna().any() else None),
})
print(summary.sort_values("missing_pct").to_string())

                             missing_pct  distinct       example
loan_id                              0.0     50000  F08Q10000071
estimated_ltv                        0.0       195           999
interest_bearing_upb                 0.0   1682515      58000.00
non_interest_bearing_upb             0.0       646          0.00
current_interest_rate                0.0       230         6.750
servicer_name                        0.0        53         OTHER
cumulative_mod_costs                 0.0    120827          0.00
remaining_months                     0.0       496           360
loan_age                             0.0       222             0
delinquency_status                   0.0       101            00
current_upb                          0.0   1682706      58000.00
period                               0.0       222        200802
mi_cancellation_flag                 0.0         1             7
current_period_mod_costs            94.7     66154        431.64
modification_flag        

In [17]:
print("estimated_ltv = 999:", round((perf_raw["estimated_ltv"] == "999").mean() * 100, 1), "%")

estimated_ltv = 999: 92.0 %


### Remaining Performance Columns

I profiled all 35 performance columns to see which are complete, which are mostly blank, and which carry no information.

The core monthly columns are complete which are balance, interest rate, loan age, remaining months, delinquency status, and servicer. Event flags such as modification_flag and step_rate_flag are about 95% blank because they are only filled when the event happens, so in cleaning a blank should mean "no event," not missing data. One time fields like zero balance dates and the loss amounts are 98 to 100% blank because they are only filled in the month a loan ends.

Two columns need special handling. mi_cancellation_flag has only one value (7, not available) because the User Guide says it is disclosed starting in 2015. estimated_ltv looks complete, but 92% of its values are 999, which the guide defines as not available. The guide says this field is only populated from April 2017, and most 2008 loans had left the data before then. In cleaning, 999 becomes null.

Most of these columns describe what happened after the loan was made, so I will not use them as model features, since that would leak future information. The performance file is used only to build the target. Model features come from the origination file and FRED data at the time of origination. In Silver I will keep all columns with clean types, and the Gold layer will select only what the model needs.

### The all years Scan
Everything so far comes from one vintage, but my models train and test across 2005 to 2022. I need to know the other 17 files have the same structure and the same quirks, because a different column count, a new status code, or more territory loans would break the Silver job. This is a quick health check across all years, not a deep dive, so it reads only the columns it needs.

In [18]:
rows = []
for year in range(2005, 2023):
    with zipfile.ZipFile(LANDING / f"sample_{year}.zip") as z:
        names = z.namelist()
        orig_name = [n for n in names if "orig" in n][0]
        perf_name = [n for n in names if "orig" not in n][0]

        with z.open(orig_name) as f:
            orig = pd.read_csv(f, sep="|", header=None, dtype=str)
        with z.open(perf_name) as f:
            perf_cols_count = len(f.readline().decode().strip().split("|"))
        with z.open(perf_name) as f:
            perf = pd.read_csv(f, sep="|", header=None, dtype=str, usecols=[0, 3])

    status = perf[3]
    states = orig[16]
    rows.append({
        "year": year,
        "orig_rows": len(orig),
        "orig_cols": orig.shape[1],
        "perf_rows": len(perf),
        "perf_cols": perf_cols_count,
        "perf_loans": perf[0].nunique(),
        "other_codes": sorted(status[~status.str.isdigit()].unique()),
        "max_status": status[status.str.isdigit()].astype(int).max(),
        "territory_loans": states.isin(["PR", "GU", "VI"]).sum(),
        "credit_9999": (orig[0] == "9999").sum(),
    })
    print(year, "done")

scan = pd.DataFrame(rows)
display(scan)

2005 done
2006 done
2007 done
2008 done
2009 done
2010 done
2011 done
2012 done
2013 done
2014 done
2015 done
2016 done
2017 done
2018 done
2019 done
2020 done
2021 done
2022 done


,year,orig_rows,orig_cols,perf_rows,perf_cols,perf_loans,other_codes,max_status,territory_loans,credit_9999
0,2005,50000,31,3880689,35,50000,[RA],99,209,41
1,2006,50000,31,3206738,35,50000,[RA],99,193,46
2,2007,50000,31,3015985,35,50000,[RA],99,228,38
3,2008,50000,31,2459814,35,50000,[RA],99,164,35
4,2009,50000,31,3166993,35,50000,[RA],99,94,1
5,2010,50000,31,3455416,35,50000,[RA],99,47,3
6,2011,50000,31,3645574,35,50000,[RA],99,25,0
7,2012,50000,31,4538907,35,50000,[RA],81,35,0
8,2013,50000,31,4228916,35,50000,[RA],83,37,1
9,2014,50000,31,3420282,35,50000,[RA],83,41,13


#### All Years Scan result
I ran the key checks on all 18 vintages (2005 to 2022). Every year has 50,000 loans, 31 origination columns, and 35 performance columns, and every loan in the performance file matches the origination file. This means one cleaning job can handle every year without special cases. RA is the only non numeric status code in every year, and XX never appears. The highest status is 99 for 2005 to 2011 and falls to 35 for 2022, because older loans went through the crisis and had more time to fall behind. Territory loans (PR, GU, VI) total 1,199, about 0.13% of all loans, so excluding them has little effect. Credit score 9999 appears in only 241 loans and will be treated as null. Across all years the performance data has about 56 million rows, which is why the cleaning step runs in Glue with PySpark.

## Cleaning Plan

These steps come from the profiling above and the Freddie Mac User Guide (October 2026). They will be done in the Silver layer with a Glue PySpark job.

### Origination file

1) Read the pipe delimited files with no header and apply the column names from config/freddie_mac_columns.yaml. All 18 vintages have 31 columns, so one layout works for every year.
2) Convert numeric columns to numbers (credit_score, mi_pct, num_units, orig_cltv, orig_dti, orig_upb, orig_ltv, orig_interest_rate, orig_loan_term, and num_borrowers).
3) Convert first_payment_date and maturity_date from YYYYMM text to dates, using the first day of the month.
4) Replace the not available codes from the User Guide with null:
   - credit_score and vantage_score: 9999
   - orig_dti, orig_ltv, orig_cltv, mi_pct: 999
   - num_units, num_borrowers, property_type: 99
   - first_time_homebuyer_flag, occupancy_status, channel, loan_purpose: 9
   - property_valuation_method: 7
   Small values like 9 or 99 in DTI and LTV are real ratios, not codes, so only the exact codes above are replaced.
5) Keep num_borrowers consistent across years. Before 2018Q2 the guide codes it as 1 or 2 (2 means more than one borrower), and from 2018Q2 it is the actual count. To compare years fairly, any value above 2 becomes 2.
6) Keep msa as null when blank. Blank means the property is not in a metropolitan area, not an error.
7) Keep property_state as is. Territory loans (PR, GU, VI) will be removed later because FRED has no data for them.
8) Add vintage (the year) and processed_at columns so every row can be traced back to its source file.

### Performance file

1) Read the files with the 35 column layout. All 18 vintages have the same layout.
2) Convert period, zero_balance_date, defect_settlement_date, and last_paid_installment_date from YYYYMM to dates.
3) Keep delinquency_status as text and add two clean columns: delinquency_months (the number, with RA as null) and is_reo (true when the status is RA). XX would become null, although it never appears in any year.
4) Convert balance, interest rate, loan age, remaining months, and cost columns to numbers.
5) Replace estimated_ltv 999 with null. The guide says this field is only filled from April 2017.
6) Replace mi_cancellation_flag 7 with null, since it means not available.
7) Keep the original flag codes as they are, and add is_modified (true when modification_flag is Y or P) and has_payment_deferral (true when payment_deferral_flag is C or P)
8) Leave the loss and cost columns as null when blank. The guide says null means not applicable for these.
9) Keep every column. Silver is a clean copy of the source, and later layers pick what the model needs.

These steps will be written as a PySpark script and tested on my laptop first, using one or two vintages from the local data folder. Once the script runs correctly and passes its checks, it will be uploaded to S3, and the Glue job will read it from there to clean all 18 vintages.